# 1. Context and data preparation

## 1.1. Experimental Setup

This cell initializes the execution environment used throughout the notebook. It sets the random seed, detects whether the notebook is running locally or in Google Colab, defines all repository, dataset, model, cache, and output paths, and reports which required resources are available. It also imports the StressID VAD utility when present and defines the deterministic filters used later to exclude technically problematic subjects and non-speech tasks.


In [ ]:
# =========================
# 1. EXPERIMENTAL SETUP
# =========================

import os
import sys
import random
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import librosa
import tqdm
import xgboost as xgb
import matplotlib.pyplot as plt

# --- 1.1 REPRODUCIBILITY ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- 1.2 EXECUTION ENVIRONMENT DETECTION ---
IN_COLAB = "google.colab" in sys.modules
EXECUTION_MODE = "colab" if IN_COLAB else "local"

# ------------------------------------------------------------
# REPOSITORY CONFIGURATION
# ------------------------------------------------------------
GITHUB_REPO_URL = "https://github.com/gabasosa/thesis-stress-recognition.git"
REPO_NAME = "thesis-stress-recognition"

# ------------------------------------------------------------
# COLAB / LOCAL BOOTSTRAP
# ------------------------------------------------------------
if IN_COLAB:
    print("Google Colab environment detected.")

    THESIS_REPO_DIR = os.path.join("/content", REPO_NAME)

    if not os.path.exists(THESIS_REPO_DIR):
        print(f"Cloning repository into {THESIS_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", GITHUB_REPO_URL, THESIS_REPO_DIR],
            check=True
        )
    else:
        print(f"Repository already present at {THESIS_REPO_DIR}")

    # Default Colab workspace root
    BASE_DIR = "/content/Tesis"

    # Optional Drive integration only if needed later
    DRIVE_DIR = "/content/drive"
    DRIVE_MOUNTED = os.path.exists(os.path.join(DRIVE_DIR, "MyDrive"))

else:
    print("Local environment detected.")

    BASE_DIR = os.path.expanduser("~/Documents/Facultad/Tesis")
    THESIS_REPO_DIR = os.path.join(BASE_DIR, "GitHub", REPO_NAME)
    DRIVE_DIR = os.path.join(BASE_DIR, "Drive")
    DRIVE_MOUNTED = os.path.exists(DRIVE_DIR)

# ============================================================
# BASE PATHS
# ============================================================

ADMIN_DIR = os.path.join(BASE_DIR, "administrative")
BACKUP_DIR = os.path.join(BASE_DIR, "backup")
DATASETS_DIR = os.path.join(BASE_DIR, "datasets")
MODELS_DIR = os.path.join(BASE_DIR, "models")
GITHUB_DIR = os.path.join(BASE_DIR, "GitHub")

# ============================================================
# DATASETS
# ============================================================

RAW_DATA_DIR = os.path.join(DATASETS_DIR, "raw")
UNPACKED_DATA_DIR = os.path.join(DATASETS_DIR, "unpacked")
PROCESSED_DATA_DIR = os.path.join(DATASETS_DIR, "processed")

STRESSID_ROOT = os.path.join(UNPACKED_DATA_DIR, "StressID")
VERBIO_ROOT = os.path.join(UNPACKED_DATA_DIR, "VerBIO_v2")

# ------------------------------------------------------------
# STRESSID DATASET METADATA PATHS (FULL DATASET COPY)
# ------------------------------------------------------------

STRESSID_LABELS_PATH = os.path.join(STRESSID_ROOT, "labels.csv")
STRESSID_LABELS_SUPPLEMENTARY_PATH = os.path.join(STRESSID_ROOT, "labels_supplementary.csv")
STRESSID_SELF_ASSESSMENTS_PATH = os.path.join(STRESSID_ROOT, "self_assessments.csv")

# ============================================================
# MODELS
# ============================================================

WAV2VEC_MODELS_DIR = os.path.join(MODELS_DIR, "wav2vec")

WAV2VEC_LARGE_PT = os.path.join(WAV2VEC_MODELS_DIR, "wav2vec_large.pt")
WAV2VEC_VOX_NEW_PT = os.path.join(WAV2VEC_MODELS_DIR, "wav2vec_vox_new.pt")

# ============================================================
# GITHUB PROJECT
# ============================================================

CONFIGS_DIR = os.path.join(THESIS_REPO_DIR, "configs")
EXPERIMENTS_DIR = os.path.join(THESIS_REPO_DIR, "experiments")
OPTUNA_DIR = os.path.join(EXPERIMENTS_DIR, "optuna")
EXTERNAL_DIR = os.path.join(THESIS_REPO_DIR, "external")
NOTEBOOKS_DIR = os.path.join(THESIS_REPO_DIR, "notebooks")
SCRIPTS_DIR = os.path.join(THESIS_REPO_DIR, "scripts")
SRC_DIR = os.path.join(THESIS_REPO_DIR, "src")

STRESSID_FORK_GITHUB_DIR = os.path.join(EXTERNAL_DIR, "stressID-fork")
VAD_DIR = os.path.join(STRESSID_FORK_GITHUB_DIR, "Feature Extraction", "audio")

# ------------------------------------------------------------
# REPOSITORY METADATA FALLBACK PATHS
# ------------------------------------------------------------

METADATA_DIR = os.path.join(THESIS_REPO_DIR, "metadata")
STRESSID_METADATA_DIR = os.path.join(METADATA_DIR, "stressID")

STRESSID_LABELS_REPO_PATH = os.path.join(STRESSID_METADATA_DIR, "labels.csv")
STRESSID_LABELS_SUPPLEMENTARY_REPO_PATH = os.path.join(STRESSID_METADATA_DIR, "labels_supplementary.csv")
STRESSID_SELF_ASSESSMENTS_REPO_PATH = os.path.join(STRESSID_METADATA_DIR, "self_assessments.csv")

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

CACHE_DIR = os.path.join(EXPERIMENTS_DIR, "cache")
EVAL_DIR = os.path.join(EXPERIMENTS_DIR, "evaluation")
IMPORTANCE_DIR = os.path.join(EXPERIMENTS_DIR, "importance")

for d in [EXPERIMENTS_DIR, OPTUNA_DIR, CACHE_DIR, EVAL_DIR, IMPORTANCE_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# OPTIONAL DRIVE FALLBACKS (COLAB ONLY)
# ============================================================
# These are only used if the default /content/Tesis paths are missing.

if IN_COLAB:
    if not os.path.exists(STRESSID_ROOT) or not os.path.exists(VERBIO_ROOT):
        print("Local Colab datasets not found under /content/Tesis.")
        print("If needed, the notebook can mount Drive and use datasets from there.")

    if not os.path.exists(WAV2VEC_LARGE_PT) and not os.path.exists(WAV2VEC_VOX_NEW_PT):
        print("Local Colab wav2vec checkpoints not found under /content/Tesis.")
        print("If needed, the notebook can mount Drive and use models from there.")

# ============================================================
# OPTIONAL: add local code to sys.path
# ============================================================

for p in [SRC_DIR, SCRIPTS_DIR, VAD_DIR]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# ============================================================
# RESOURCE FLAGS
# ============================================================

HAS_THESIS_REPO = os.path.exists(THESIS_REPO_DIR)
HAS_EXPERIMENTS_DIR = os.path.exists(EXPERIMENTS_DIR)
HAS_STRESSID_DATASET = os.path.exists(STRESSID_ROOT)
HAS_VERBIO_DATASET = os.path.exists(VERBIO_ROOT)
HAS_WAV2VEC_LARGE = os.path.exists(WAV2VEC_LARGE_PT)
HAS_WAV2VEC_VOX_NEW = os.path.exists(WAV2VEC_VOX_NEW_PT)
HAS_VAD_DIR = os.path.exists(VAD_DIR)

HAS_STRESSID_LABELS = os.path.exists(STRESSID_LABELS_PATH)
HAS_STRESSID_LABELS_SUPPLEMENTARY = os.path.exists(STRESSID_LABELS_SUPPLEMENTARY_PATH)
HAS_STRESSID_SELF_ASSESSMENTS = os.path.exists(STRESSID_SELF_ASSESSMENTS_PATH)

HAS_STRESSID_LABELS_REPO = os.path.exists(STRESSID_LABELS_REPO_PATH)
HAS_STRESSID_LABELS_SUPPLEMENTARY_REPO = os.path.exists(STRESSID_LABELS_SUPPLEMENTARY_REPO_PATH)
HAS_STRESSID_SELF_ASSESSMENTS_REPO = os.path.exists(STRESSID_SELF_ASSESSMENTS_REPO_PATH)

# ============================================================
# SANITY REPORT
# ============================================================

print("====================================================")
print(" EXPERIMENTAL ENVIRONMENT")
print("====================================================")
print("EXECUTION_MODE:", EXECUTION_MODE)
print("IN_COLAB:", IN_COLAB)
print("BASE_DIR:", BASE_DIR)
print("THESIS_REPO_DIR:", THESIS_REPO_DIR)
print("DATASETS_DIR:", DATASETS_DIR)
print("STRESSID_ROOT:", STRESSID_ROOT)
print("VERBIO_ROOT:", VERBIO_ROOT)
print("WAV2VEC_MODELS_DIR:", WAV2VEC_MODELS_DIR)
print("VAD_DIR:", VAD_DIR)
print("CACHE_DIR:", CACHE_DIR)
print("OPTUNA_DIR:", OPTUNA_DIR)
print("DRIVE_MOUNTED:", DRIVE_MOUNTED)
print("====================================================")
print("HAS_THESIS_REPO:", HAS_THESIS_REPO)
print("HAS_EXPERIMENTS_DIR:", HAS_EXPERIMENTS_DIR)
print("HAS_STRESSID_DATASET:", HAS_STRESSID_DATASET)
print("HAS_VERBIO_DATASET:", HAS_VERBIO_DATASET)
print("HAS_WAV2VEC_LARGE:", HAS_WAV2VEC_LARGE)
print("HAS_WAV2VEC_VOX_NEW:", HAS_WAV2VEC_VOX_NEW)
print("HAS_VAD_DIR:", HAS_VAD_DIR)
print("====================================================")
print("STRESSID_LABELS_PATH:", STRESSID_LABELS_PATH)
print("STRESSID_LABELS_SUPPLEMENTARY_PATH:", STRESSID_LABELS_SUPPLEMENTARY_PATH)
print("STRESSID_SELF_ASSESSMENTS_PATH:", STRESSID_SELF_ASSESSMENTS_PATH)
print("STRESSID_LABELS_REPO_PATH:", STRESSID_LABELS_REPO_PATH)
print("STRESSID_LABELS_SUPPLEMENTARY_REPO_PATH:", STRESSID_LABELS_SUPPLEMENTARY_REPO_PATH)
print("STRESSID_SELF_ASSESSMENTS_REPO_PATH:", STRESSID_SELF_ASSESSMENTS_REPO_PATH)
print("====================================================")
print("HAS_STRESSID_LABELS:", HAS_STRESSID_LABELS)
print("HAS_STRESSID_LABELS_SUPPLEMENTARY:", HAS_STRESSID_LABELS_SUPPLEMENTARY)
print("HAS_STRESSID_SELF_ASSESSMENTS:", HAS_STRESSID_SELF_ASSESSMENTS)
print("HAS_STRESSID_LABELS_REPO:", HAS_STRESSID_LABELS_REPO)
print("HAS_STRESSID_LABELS_SUPPLEMENTARY_REPO:", HAS_STRESSID_LABELS_SUPPLEMENTARY_REPO)
print("HAS_STRESSID_SELF_ASSESSMENTS_REPO:", HAS_STRESSID_SELF_ASSESSMENTS_REPO)
print("====================================================")

assert HAS_THESIS_REPO, f"Missing THESIS_REPO_DIR: {THESIS_REPO_DIR}"
assert HAS_EXPERIMENTS_DIR, f"Missing EXPERIMENTS_DIR: {EXPERIMENTS_DIR}"

# ============================================================
# OPTIONAL IMPORTS
# ============================================================

if HAS_VAD_DIR:
    import Speech_silence_vad
    print("Speech_silence_vad imported successfully.")
else:
    print("Warning: Speech_silence_vad was not imported because VAD_DIR was not found.")

# --- 1.3 DATA INTEGRITY FILTERS ---
TECH_BLACKLIST = ['hh2e', 'wfsl', '37ir', 'hvpa', 'ql3b', 'r0a3', 'uyrl', 'dmbd', 'qx2o', 'f6q3']
TALKING_TASKS = ['Counting1', 'Counting2', 'Counting3', 'Math', 'Stroop', 'Reading', 'Speaking']

print("Experimental environment initialized.")

RUNNING_WITH_FULL_DATA = HAS_STRESSID_DATASET and HAS_WAV2VEC_LARGE
RUNNING_WITH_CACHE_ONLY = not RUNNING_WITH_FULL_DATA

print("RUNNING_WITH_FULL_DATA:", RUNNING_WITH_FULL_DATA)
print("RUNNING_WITH_CACHE_ONLY:", RUNNING_WITH_CACHE_ONLY)

## 1.2. Corpus Ingestion and Filtering

This cell builds the working StressID speech table from `labels.csv`. The `subject/task` identifier is parsed into separate subject and task fields, and each row is mapped to its corresponding audio file under the StressID directory structure. Non-verbal tasks and recordings associated with known technical problems are excluded before model training. The resulting dataframe, `df_sid`, contains one row per retained speech recording, with its file path, subject identifier, task name, and binary stress label.


In [ ]:
def resolve_existing_path(candidates, description, required=True):
    """Return the first existing path from a list of candidates."""
    for candidate in candidates:
        if os.path.isfile(candidate):
            print(f"Using {description}: {candidate}")
            return candidate

    if required:
        candidate_list = "\n".join(f"- {candidate}" for candidate in candidates)
        raise FileNotFoundError(
            f"{description} not found in any expected location:\n{candidate_list}"
        )

    print(f"{description} not found. Continuing without it.")
    return None


def get_clean_stressid():
    # ------------------------------------------------------------
    # Resolve metadata paths
    # ------------------------------------------------------------
    labels_path = resolve_existing_path(
        [STRESSID_LABELS_PATH, STRESSID_LABELS_REPO_PATH],
        "labels.csv",
        required=True
    )

    # These companion files are reported for traceability, but the current
    # supervised target is read from labels.csv.
    _ = resolve_existing_path(
        [STRESSID_LABELS_SUPPLEMENTARY_PATH, STRESSID_LABELS_SUPPLEMENTARY_REPO_PATH],
        "labels_supplementary.csv",
        required=False
    )

    _ = resolve_existing_path(
        [STRESSID_SELF_ASSESSMENTS_PATH, STRESSID_SELF_ASSESSMENTS_REPO_PATH],
        "self_assessments.csv",
        required=False
    )

    # ------------------------------------------------------------
    # Load and validate labels table
    # ------------------------------------------------------------
    df = pd.read_csv(labels_path)

    required_columns = {"subject/task", "binary-stress"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns in labels.csv: {sorted(missing_columns)}")

    key = (
        df["subject/task"]
        .astype(str)
        .str.replace("/", "_", regex=False)
    )

    subject_task = key.str.split("_", n=1, expand=True)

    if subject_task.shape[1] < 2:
        raise ValueError("Could not parse 'subject/task' into subject and task fields.")

    df["subject"] = subject_task[0]
    df["task"] = subject_task[1]

    invalid_key_mask = df["task"].isna() | (df["task"].astype(str).str.len() == 0)
    if invalid_key_mask.any():
        n_invalid = int(invalid_key_mask.sum())
        print(f"Warning: dropping {n_invalid} rows with invalid subject/task identifiers.")
        df = df.loc[~invalid_key_mask].copy()

    # ------------------------------------------------------------
    # Deterministic filtering
    # ------------------------------------------------------------
    mask = (
        ~df["subject"].isin(TECH_BLACKLIST)
        & df["task"].isin(TALKING_TASKS)
    )
    df = df.loc[mask].copy()

    # ------------------------------------------------------------
    # Reconstruct audio paths and drop unavailable files
    # ------------------------------------------------------------
    df["filepath"] = df.apply(
        lambda row: os.path.join(
            STRESSID_ROOT,
            "Audio",
            row["subject"],
            f"{row['subject']}_{row['task']}.wav"
        ),
        axis=1
    )

    missing_mask = ~df["filepath"].apply(os.path.isfile)
    n_missing = int(missing_mask.sum())

    if n_missing > 0:
        missing_subjects = sorted(df.loc[missing_mask, "subject"].unique().tolist())
        print(f"Warning: {n_missing} expected audio files were not found. Dropping those rows.")
        print(f"Subjects with missing files: {missing_subjects}")
        df = df.loc[~missing_mask].copy()

    df["binary-stress"] = df["binary-stress"].astype(int)

    return (
        df[["filepath", "subject", "task", "binary-stress"]]
        .reset_index(drop=True)
    )


df_sid = get_clean_stressid()

print("====================================================")
print(" STRESSID SPEECH TABLE READY")
print("====================================================")
print(f"Samples: {len(df_sid)}")
print(f"Subjects: {df_sid['subject'].nunique()}")
print(f"Tasks: {sorted(df_sid['task'].unique().tolist())}")
print(f"Positive class ratio (y=1): {df_sid['binary-stress'].mean():.3f}")
print("====================================================")


# 2. Handcrafted-feature pipeline

## 2.1. Audio Feature Extraction Methodology

This cell defines the handcrafted acoustic feature extractor used for the StressID baseline experiments. Each waveform is loaded, passed through the StressID voice activity detection (VAD) routine, and converted into a fixed-length 140-dimensional representation. The representation concatenates temporal means and standard deviations of cepstral, spectral, tonal, zero-crossing, and rhythmic descriptors. A dimensionality assertion is kept at the end of the function so that changes in library behavior or feature construction are detected immediately.


In [ ]:
def apply_stressid_vad(speech, sr, min_duration_s=0.10):
    """
    Apply the StressID VAD routine and fall back to the original signal if
    VAD returns an empty or unrealistically short segment.
    """
    if "Speech_silence_vad" not in globals():
        raise RuntimeError(
            "Speech_silence_vad is not available. Run the experimental setup cell "
            "and verify that VAD_DIR points to the StressID audio feature code."
        )

    speech_vad = Speech_silence_vad.silence_handler(
        speech,
        sr,
        fl=int(20 / 1000 * sr),
        fs=int(5 / 1000 * sr),
        max_thres_below=40,
        min_thres=-55,
        shortest_len_in_ms=50,
        flag_output=1
    )

    speech_vad = np.asarray(speech_vad, dtype=np.float32)

    min_samples = int(min_duration_s * sr)
    if speech_vad.size < min_samples:
        return np.asarray(speech, dtype=np.float32)

    return speech_vad


def HandcraftedAudioFeatures(audio_path):
    speech, sr = librosa.load(audio_path, sr=None, mono=True)
    speech_vad = apply_stressid_vad(speech, sr)

    # ------------------------------------------------------------
    # Cepstral descriptors
    # ------------------------------------------------------------
    mfcc = librosa.feature.mfcc(y=speech_vad, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    mfcc_delta = librosa.feature.delta(mfcc)
    mfcc_delta_mean = np.mean(mfcc_delta, axis=1)
    mfcc_delta_std = np.std(mfcc_delta, axis=1)

    mfcc_delta_delta = librosa.feature.delta(mfcc_delta)
    mfcc_delta_delta_mean = np.mean(mfcc_delta_delta, axis=1)
    mfcc_delta_delta_std = np.std(mfcc_delta_delta, axis=1)

    # ------------------------------------------------------------
    # Spectral descriptors
    # ------------------------------------------------------------
    spectral_centroid = librosa.feature.spectral_centroid(y=speech_vad, sr=sr)
    spectral_centroid_mean = np.mean(spectral_centroid, axis=1)
    spectral_centroid_std = np.std(spectral_centroid, axis=1)

    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=speech_vad, sr=sr)
    spectral_bandwidth_mean = np.mean(spectral_bandwidth, axis=1)
    spectral_bandwidth_std = np.std(spectral_bandwidth, axis=1)

    spectral_flatness = librosa.feature.spectral_flatness(y=speech_vad)
    spectral_flatness_mean = np.mean(spectral_flatness, axis=1)
    spectral_flatness_std = np.std(spectral_flatness, axis=1)

    stft_magnitude = np.abs(librosa.stft(speech_vad))

    spectral_contrast = librosa.feature.spectral_contrast(S=stft_magnitude, sr=sr)
    spectral_contrast_mean = np.mean(spectral_contrast, axis=1)
    spectral_contrast_std = np.std(spectral_contrast, axis=1)

    spectral_rolloff = librosa.feature.spectral_rolloff(
        y=speech_vad,
        sr=sr,
        roll_percent=0.85
    )
    spectral_rolloff_mean = np.mean(spectral_rolloff, axis=1)
    spectral_rolloff_std = np.std(spectral_rolloff, axis=1)

    # ------------------------------------------------------------
    # Tonal, zero-crossing, and rhythmic descriptors
    # ------------------------------------------------------------
    harmonic_signal = librosa.effects.harmonic(speech_vad)

    tonnetz = librosa.feature.tonnetz(y=harmonic_signal, sr=sr)
    tonnetz_mean = np.mean(tonnetz, axis=1)
    tonnetz_std = np.std(tonnetz, axis=1)

    zcr = librosa.feature.zero_crossing_rate(speech_vad)
    zcr_mean = np.mean(zcr, axis=1)
    zcr_std = np.std(zcr, axis=1)

    tempogram = librosa.feature.tempogram(y=speech_vad, sr=sr)
    tempogram_ratio = librosa.feature.tempogram_ratio(tg=tempogram, sr=sr)
    tempogram_ratio_mean = np.mean(tempogram_ratio, axis=1)
    tempogram_ratio_std = np.std(tempogram_ratio, axis=1)

    # ------------------------------------------------------------
    # Concatenation order follows the StressID handcrafted pipeline
    # ------------------------------------------------------------
    handcrafted_features = np.concatenate([
        mfcc_mean, mfcc_std,
        mfcc_delta_mean, mfcc_delta_std,
        mfcc_delta_delta_mean, mfcc_delta_delta_std,
        spectral_centroid_mean, spectral_centroid_std,
        spectral_bandwidth_mean, spectral_bandwidth_std,
        spectral_flatness_mean, spectral_flatness_std,
        spectral_contrast_mean, spectral_contrast_std,
        spectral_rolloff_mean, spectral_rolloff_std,
        tonnetz_mean, tonnetz_std,
        zcr_mean, zcr_std,
        tempogram_ratio_mean, tempogram_ratio_std
    ], axis=0).astype(np.float32)

    assert handcrafted_features.shape[0] == 140, (
        f"Unexpected handcrafted feature size: {handcrafted_features.shape[0]}"
    )

    return handcrafted_features


## 2.2. Dataset-Level Feature Matrix Construction and Caching

This cell converts the filtered StressID table into the supervised-learning arrays used by the handcrafted-feature experiments. For each retained recording, the 140-dimensional handcrafted extractor produces one feature vector. These vectors are stacked into `X_audio`, while the binary labels and subject identifiers are stored in `y_audio` and `groups_audio`, respectively. The generated matrix is cached as a parquet file so that later notebook runs can reuse the same features without repeating the audio extraction step. When a cache is already available, local execution asks whether to reuse it; Colab execution expects the cache to be present.


In [ ]:
# ============================================================
# 2.2. Dataset-Level Feature Matrix Construction and Caching
# ============================================================

FEATURE_CACHE = os.path.join(CACHE_DIR, "stressid_audio_features_handcrafted.parquet")
META_FILE = os.path.join(CACHE_DIR, "stressid_audio_features_handcrafted_meta.txt")


def ordered_feature_columns(df):
    """Return feature columns f_0, f_1, ... sorted by their numeric index."""
    feature_cols = [col for col in df.columns if col.startswith("f_")]

    if not feature_cols:
        raise ValueError("No feature columns with prefix 'f_' were found.")

    return sorted(feature_cols, key=lambda col: int(col.split("_", 1)[1]))


def load_feature_cache(cache_path, label_col="label", group_col="subject"):
    df_cached = pd.read_parquet(cache_path)
    feature_cols = ordered_feature_columns(df_cached)

    required_cols = {label_col, group_col}
    missing_cols = required_cols.difference(df_cached.columns)
    if missing_cols:
        raise ValueError(f"Missing required cache columns: {sorted(missing_cols)}")

    X = df_cached[feature_cols].values.astype(np.float32)
    y = df_cached[label_col].values.astype(np.int64)
    groups = df_cached[group_col].values.astype(object)

    return df_cached, X, y, groups


cache_exists = os.path.exists(FEATURE_CACHE)
meta_exists = os.path.exists(META_FILE)

use_cached_features = False
compute_features_from_scratch = False

# ------------------------------------------------------------
# Decide whether to use cache or recompute
# ------------------------------------------------------------
if cache_exists:
    print("====================================================")
    print(" EXISTING HANDCRAFTED FEATURE CACHE DETECTED")
    print("====================================================")

    if meta_exists and not IN_COLAB:
        with open(META_FILE, "r", encoding="utf-8") as f:
            print(f.read())

    if IN_COLAB:
        print("Colab environment detected. Using cached handcrafted features automatically.")
        use_cached_features = True
    else:
        user_input = input(
            "Reuse cached handcrafted features instead of recomputing from scratch? (y/n): "
        ).strip().lower()

        if user_input == "y":
            print("Using existing cached feature dataset.")
            use_cached_features = True
        else:
            print(
                "Cached handcrafted features will be ignored. "
                "A new extraction will overwrite the existing cache."
            )
            compute_features_from_scratch = True

else:
    if IN_COLAB:
        raise FileNotFoundError(
            "No cached handcrafted feature file was found in Colab.\n"
            f"Expected cache path: {FEATURE_CACHE}\n"
            "Colab execution requires the cached parquet file because recomputation is disabled there."
        )

    print("No cached handcrafted features found. Features will be extracted from scratch.")
    compute_features_from_scratch = True

# ------------------------------------------------------------
# Load cached features
# ------------------------------------------------------------
if use_cached_features:
    df_features, X_audio, y_audio, groups_audio = load_feature_cache(FEATURE_CACHE)

    print("====================================================")
    print(" HANDCRAFTED FEATURE DATASET READY")
    print("====================================================")
    print("Source: cache")
    print(f"Samples: {X_audio.shape[0]}")
    print(f"Feature dim: {X_audio.shape[1]}")
    print(f"Positive class ratio (y=1): {float(np.mean(y_audio)):.3f}")
    print("====================================================")

# ------------------------------------------------------------
# Compute features from scratch
# ------------------------------------------------------------
elif compute_features_from_scratch:
    required_cols = {"filepath", "subject", "binary-stress"}

    assert "df_sid" in locals(), "df_sid not found. Run the corpus-ingestion cell first."
    assert required_cols.issubset(set(df_sid.columns)), (
        f"df_sid must contain columns: {sorted(required_cols)}"
    )
    assert "HandcraftedAudioFeatures" in locals(), (
        "HandcraftedAudioFeatures not found. Run the handcrafted feature-definition cell first."
    )

    X_audio_list = []
    y_audio_list = []
    groups_audio_list = []
    failed_files = []

    for _, row in tqdm.tqdm(
        df_sid.iterrows(),
        total=len(df_sid),
        desc="Extracting handcrafted audio features"
    ):
        audio_path = row["filepath"]
        label = int(row["binary-stress"])
        subject = row["subject"]

        try:
            features = HandcraftedAudioFeatures(audio_path)
            X_audio_list.append(features)
            y_audio_list.append(label)
            groups_audio_list.append(subject)
        except Exception as exc:
            failed_files.append((audio_path, str(exc)))

    n_ok = len(X_audio_list)
    n_fail = len(failed_files)

    print("====================================================")
    print(" HANDCRAFTED FEATURE EXTRACTION SUMMARY")
    print("====================================================")
    print(f"Total rows in df_sid: {len(df_sid)}")
    print(f"Successful extractions: {n_ok}")
    print(f"Failed extractions: {n_fail}")

    if n_fail > 0:
        print("\nFirst 10 failures:")
        for failed_path, error_message in failed_files[:10]:
            print(f" - {failed_path}\n   {error_message}")

    assert n_ok > 0, "No handcrafted features were extracted successfully. See failures above."

    X_audio = np.vstack(X_audio_list).astype(np.float32)
    y_audio = np.array(y_audio_list, dtype=np.int64)
    groups_audio = np.array(groups_audio_list, dtype=object)

    feature_cols = [f"f_{i}" for i in range(X_audio.shape[1])]
    df_features = pd.DataFrame(X_audio, columns=feature_cols)
    df_features["label"] = y_audio
    df_features["subject"] = groups_audio

    df_features.to_parquet(FEATURE_CACHE, index=False)

    extraction_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    meta_text = (
        f"Last extraction timestamp: {extraction_timestamp}\n"
        f"Samples: {len(df_features)}\n"
        f"Feature dimension: {X_audio.shape[1]}\n"
        f"Positive ratio: {float(np.mean(y_audio)):.3f}\n"
        f"Failed files: {len(failed_files)}\n"
        f"Cache file: {FEATURE_CACHE}\n"
    )

    with open(META_FILE, "w", encoding="utf-8") as f:
        f.write(meta_text)

    print("\n====================================================")
    print(" HANDCRAFTED FEATURE DATASET READY")
    print("====================================================")
    print("Source: fresh extraction")
    print(f"Samples: {X_audio.shape[0]}")
    print(f"Feature dim: {X_audio.shape[1]}")
    print(f"Positive class ratio (y=1): {float(np.mean(y_audio)):.3f}")
    print(f"Failed files: {len(failed_files)}")
    print("====================================================")
    print(meta_text)


## 2.3. Baseline Evaluation Protocol

This cell defines reusable cross-validation helpers for the handcrafted baseline classifiers. The local baseline uses k-nearest neighbors with k = 5 and a linear support vector machine, both wrapped in scikit-learn pipelines so that feature scaling is fitted only on each training fold. Two validation schemes are supported: stratified K-fold, which preserves class balance across folds, and GroupKFold, which keeps all recordings from the same subject in the same fold for subject-independent evaluation.

In [ ]:
# ============================================================
# 2.3. Baseline Evaluation Protocol
# ============================================================

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

n_folds = 5
KNN_N_NEIGHBORS = 5
KNN_MODEL_LABEL = f"kNN (k={KNN_N_NEIGHBORS})"
SVM_KERNEL = "linear"
SVM_MODEL_LABEL = f"SVM ({SVM_KERNEL})"


def build_baseline_models(k_neighbors=KNN_N_NEIGHBORS, svm_kernel=SVM_KERNEL):
    """Create the baseline classifiers used for local audio-feature evaluation."""
    return {
        "knn": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=k_neighbors))
        ]),
        "svm": Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(kernel=svm_kernel, C=1.0, class_weight="balanced"))
        ])
    }


def summarize_fold_scores(f1_scores, acc_scores, bacc_scores):
    return {
        "f1_mean": float(np.mean(f1_scores)),
        "f1_std": float(np.std(f1_scores, ddof=1)),
        "acc_mean": float(np.mean(acc_scores)),
        "acc_std": float(np.std(acc_scores, ddof=1)),
        "bacc_mean": float(np.mean(bacc_scores)),
        "bacc_std": float(np.std(bacc_scores, ddof=1)),
    }


def evaluate_models_cv(X, y, models, cv, groups=None):
    """
    Evaluate one or more sklearn-compatible estimators under a given CV splitter.

    A fresh clone of each estimator is fitted in each fold to avoid carrying
    fitted state across folds.
    """
    X = np.asarray(X)
    y = np.asarray(y)
    groups = None if groups is None else np.asarray(groups)

    results = {}

    for model_name, estimator in models.items():
        f1_scores = []
        acc_scores = []
        bacc_scores = []

        if groups is None:
            split_iter = cv.split(X, y)
        else:
            split_iter = cv.split(X, y, groups)

        for train_idx, test_idx in split_iter:
            model = clone(estimator)

            X_train = X[train_idx]
            X_test = X[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            f1_scores.append(f1_score(y_test, y_pred, average="weighted"))
            acc_scores.append(accuracy_score(y_test, y_pred))
            bacc_scores.append(balanced_accuracy_score(y_test, y_pred))

        results[model_name] = summarize_fold_scores(
            f1_scores=f1_scores,
            acc_scores=acc_scores,
            bacc_scores=bacc_scores
        )

    return results


def flatten_baseline_results(results, n_splits, X, y, groups=None, seed=None):
    """Convert generic model results into the historical flat key format."""
    flat = {}

    for model_key, metrics in results.items():
        for metric_key, value in metrics.items():
            flat[f"{model_key}_{metric_key}"] = value

    flat["n_splits"] = int(n_splits)
    flat["n_samples"] = int(np.asarray(X).shape[0])
    flat["pos_ratio"] = float(np.mean(y))

    if seed is not None:
        flat["seed"] = int(seed)

    if groups is not None:
        flat["n_groups"] = int(len(np.unique(groups)))

    return flat


def run_audio_benchmark(X, y, n_splits=n_folds, seed=SEED):
    """Evaluate the local handcrafted baselines with stratified K-fold CV."""
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    results = evaluate_models_cv(
        X=X,
        y=y,
        models=build_baseline_models(k_neighbors=KNN_N_NEIGHBORS, svm_kernel=SVM_KERNEL),
        cv=cv,
        groups=None
    )

    return flatten_baseline_results(
        results=results,
        n_splits=n_splits,
        X=X,
        y=y,
        seed=seed
    )


def run_audio_benchmark_groupkfold(X, y, groups, n_splits=n_folds):
    """Evaluate the local handcrafted baselines with subject-independent CV."""
    cv = GroupKFold(n_splits=n_splits)

    results = evaluate_models_cv(
        X=X,
        y=y,
        models=build_baseline_models(k_neighbors=KNN_N_NEIGHBORS, svm_kernel=SVM_KERNEL),
        cv=cv,
        groups=groups
    )

    return flatten_baseline_results(
        results=results,
        n_splits=n_splits,
        X=X,
        y=y,
        groups=groups
    )

## 2.4. Baseline Execution and Cross-Validated Results

This cell runs the handcrafted-feature baselines under both validation schemes. The stratified results are reported alongside the values published for the StressID audio handcrafted baseline, making it easier to check whether the local implementation behaves consistently with the reference benchmark. The subject-independent GroupKFold results are reported as an additional, stricter experiment because they evaluate generalization to unseen speakers. Weighted F1-score, accuracy, and balanced accuracy are retained for later comparison tables.


In [ ]:
# ============================================================
# 2.4. Baseline Execution and Cross-Validated Results
# ============================================================

def fmt_short(mean, std):
    return f"{mean:.2f} ± {std:.2f}"


knn_label = KNN_MODEL_LABEL if "KNN_MODEL_LABEL" in globals() else "kNN (k=5)"
svm_label = SVM_MODEL_LABEL if "SVM_MODEL_LABEL" in globals() else "SVM (linear)"


# StressID paper baseline values for 2-class audio handcrafted features.
PAPER_RESULTS = {
    "knn": {"f1": (0.67, 0.06), "acc": (0.60, 0.05)},
    "svm": {"f1": (0.61, 0.06), "acc": (0.54, 0.03)},
}

assert "X_audio" in locals(), "X_audio not found. Run Section 2.2 first."
assert "y_audio" in locals(), "y_audio not found. Run Section 2.2 first."
assert "groups_audio" in locals(), "groups_audio not found. Run Section 2.2 first."

# ------------------------------------------------------------
# Run evaluations
# ------------------------------------------------------------
hc_baseline_strat_results = run_audio_benchmark(
    X_audio,
    y_audio,
    n_splits=n_folds,
    seed=SEED
)

hc_baseline_group_results = run_audio_benchmark_groupkfold(
    X_audio,
    y_audio,
    groups_audio,
    n_splits=n_folds
)

# ------------------------------------------------------------
# Paper-comparable stratified results
# ------------------------------------------------------------
print("==========================================================================================")
print(" STRESSID AUDIO HANDCRAFTED BASELINE: STRATIFIED CV")
print("==========================================================================================")
print(f"{'Baseline':<36} {'Local F1-score':<18} {'Paper F1-score':<18} {'Local Acc.':<18} {'Paper Acc.':<18}")
print("------------------------------------------------------------------------------------------")

print(
    f"{f'Audio HC features + {knn_label}':<36} "
    f"{fmt_short(hc_baseline_strat_results['knn_f1_mean'], hc_baseline_strat_results['knn_f1_std']):<18} "
    f"{fmt_short(*PAPER_RESULTS['knn']['f1']):<18} "
    f"{fmt_short(hc_baseline_strat_results['knn_acc_mean'], hc_baseline_strat_results['knn_acc_std']):<18} "
    f"{fmt_short(*PAPER_RESULTS['knn']['acc']):<18}"
)

print(
    f"{f'Audio HC features + {svm_label}':<36} "
    f"{fmt_short(hc_baseline_strat_results['svm_f1_mean'], hc_baseline_strat_results['svm_f1_std']):<18} "
    f"{fmt_short(*PAPER_RESULTS['svm']['f1']):<18} "
    f"{fmt_short(hc_baseline_strat_results['svm_acc_mean'], hc_baseline_strat_results['svm_acc_std']):<18} "
    f"{fmt_short(*PAPER_RESULTS['svm']['acc']):<18}"
)

print("==========================================================================================")

# ------------------------------------------------------------
# Subject-independent results
# ------------------------------------------------------------
print("\nSubject-independent evaluation (additional local experiment)")
print("==========================================================================================")
print(f"{'Baseline':<36} {'GroupKFold F1-score':<22} {'GroupKFold Acc.':<22} {'GroupKFold BAcc.':<22}")
print("------------------------------------------------------------------------------------------")

print(
    f"{f'Audio HC features + {knn_label}':<36} "
    f"{fmt_short(hc_baseline_group_results['knn_f1_mean'], hc_baseline_group_results['knn_f1_std']):<22} "
    f"{fmt_short(hc_baseline_group_results['knn_acc_mean'], hc_baseline_group_results['knn_acc_std']):<22} "
    f"{fmt_short(hc_baseline_group_results['knn_bacc_mean'], hc_baseline_group_results['knn_bacc_std']):<22}"
)

print(
    f"{f'Audio HC features + {svm_label}':<36} "
    f"{fmt_short(hc_baseline_group_results['svm_f1_mean'], hc_baseline_group_results['svm_f1_std']):<22} "
    f"{fmt_short(hc_baseline_group_results['svm_acc_mean'], hc_baseline_group_results['svm_acc_std']):<22} "
    f"{fmt_short(hc_baseline_group_results['svm_bacc_mean'], hc_baseline_group_results['svm_bacc_std']):<22}"
)

print("==========================================================================================")

# ------------------------------------------------------------
# Diagnostic balanced accuracy summary
# ------------------------------------------------------------
print("\nDiagnostic metric summary")
print("------------------------------------------------------------------------------------------")
print(f"{knn_label} Balanced Accuracy  (Stratified CV): {fmt_short(hc_baseline_strat_results['knn_bacc_mean'], hc_baseline_strat_results['knn_bacc_std'])}")
print(f"SVM Balanced Accuracy  (Stratified CV): {fmt_short(hc_baseline_strat_results['svm_bacc_mean'], hc_baseline_strat_results['svm_bacc_std'])}")
print(f"{knn_label} Balanced Accuracy  (GroupKFold):    {fmt_short(hc_baseline_group_results['knn_bacc_mean'], hc_baseline_group_results['knn_bacc_std'])}")
print(f"SVM Balanced Accuracy  (GroupKFold):    {fmt_short(hc_baseline_group_results['svm_bacc_mean'], hc_baseline_group_results['svm_bacc_std'])}")

# ------------------------------------------------------------
# Export local baseline results
# ------------------------------------------------------------
hc_baseline_eval_path = os.path.join(
    EVAL_DIR,
    "stressid_hc_baseline_evaluation_summary.csv"
)

eval_rows = [
    {
        "feature_type": "Handcrafted",
        "model": knn_label,
        "cv_scheme": "stratifiedkfold",
        "f1_mean": hc_baseline_strat_results["knn_f1_mean"],
        "f1_std": hc_baseline_strat_results["knn_f1_std"],
        "acc_mean": hc_baseline_strat_results["knn_acc_mean"],
        "acc_std": hc_baseline_strat_results["knn_acc_std"],
        "bacc_mean": hc_baseline_strat_results["knn_bacc_mean"],
        "bacc_std": hc_baseline_strat_results["knn_bacc_std"],
    },
    {
        "feature_type": "Handcrafted",
        "model": svm_label,
        "cv_scheme": "stratifiedkfold",
        "f1_mean": hc_baseline_strat_results["svm_f1_mean"],
        "f1_std": hc_baseline_strat_results["svm_f1_std"],
        "acc_mean": hc_baseline_strat_results["svm_acc_mean"],
        "acc_std": hc_baseline_strat_results["svm_acc_std"],
        "bacc_mean": hc_baseline_strat_results["svm_bacc_mean"],
        "bacc_std": hc_baseline_strat_results["svm_bacc_std"],
    },
    {
        "feature_type": "Handcrafted",
        "model": knn_label,
        "cv_scheme": "groupkfold",
        "f1_mean": hc_baseline_group_results["knn_f1_mean"],
        "f1_std": hc_baseline_group_results["knn_f1_std"],
        "acc_mean": hc_baseline_group_results["knn_acc_mean"],
        "acc_std": hc_baseline_group_results["knn_acc_std"],
        "bacc_mean": hc_baseline_group_results["knn_bacc_mean"],
        "bacc_std": hc_baseline_group_results["knn_bacc_std"],
    },
    {
        "feature_type": "Handcrafted",
        "model": svm_label,
        "cv_scheme": "groupkfold",
        "f1_mean": hc_baseline_group_results["svm_f1_mean"],
        "f1_std": hc_baseline_group_results["svm_f1_std"],
        "acc_mean": hc_baseline_group_results["svm_acc_mean"],
        "acc_std": hc_baseline_group_results["svm_acc_std"],
        "bacc_mean": hc_baseline_group_results["svm_bacc_mean"],
        "bacc_std": hc_baseline_group_results["svm_bacc_std"],
    }
]

df_hc_baseline_eval = pd.DataFrame(eval_rows)
df_hc_baseline_eval.to_csv(hc_baseline_eval_path, index=False)

print("\nSaved:")
print(f"- {hc_baseline_eval_path}")


The stratified protocol is useful for checking whether the local handcrafted pipeline is broadly consistent with the reported StressID baseline. The subject-independent protocol is more demanding: if its scores drop substantially, this suggests that part of the stratified performance may be driven by speaker-dependent patterns rather than stress-related cues that transfer reliably to unseen subjects.


## 2.5. XGBoost Optimization with Optuna

This cell loads or runs an Optuna search for an XGBoost classifier trained on the 140-dimensional handcrafted representation. Each trial samples a tree-based configuration and evaluates it with the selected cross-validation scheme, using mean weighted F1-score as the optimization objective. Class imbalance is handled inside each training fold through `scale_pos_weight`.

The default search scheme is subject-independent GroupKFold, which makes the selected hyperparameters better aligned with the unseen-speaker setting. Cached Optuna results are reused when available, especially in Colab, where long optimization runs are intentionally avoided. The best hyperparameter set is stored for the subsequent evaluation cell.


In [ ]:
# ============================================================
# 2.5. XGBoost Hyperparameter Loading / Optuna Optimization
# ============================================================

import logging
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score

assert "X_audio" in locals(), "X_audio not found. Run the handcrafted feature matrix cell first."
assert "y_audio" in locals(), "y_audio not found. Run the handcrafted feature matrix cell first."

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
N_TRIALS = 50
OPTUNA_CV_SCHEME = "groupkfold"  # options: "groupkfold", "stratifiedkfold"

if OPTUNA_CV_SCHEME not in {"groupkfold", "stratifiedkfold"}:
    raise ValueError("OPTUNA_CV_SCHEME must be either 'groupkfold' or 'stratifiedkfold'.")

USE_GROUPKFOLD = OPTUNA_CV_SCHEME == "groupkfold"

X_audio_np = np.asarray(X_audio, dtype=np.float32)
y_audio_np = np.asarray(y_audio, dtype=np.int64)

if USE_GROUPKFOLD:
    assert "groups_audio" in locals(), "groups_audio not found, but OPTUNA_CV_SCHEME='groupkfold'."
    groups_audio_np = np.asarray(groups_audio)

cv_suffix = OPTUNA_CV_SCHEME

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------
optuna_trials_path = os.path.join(
    OPTUNA_DIR,
    f"stressid_xgb_handcrafted_optuna_trials_{cv_suffix}.csv"
)

best_params_path = os.path.join(
    OPTUNA_DIR,
    f"stressid_xgb_handcrafted_best_params_{cv_suffix}.csv"
)

trials_exist = os.path.exists(optuna_trials_path)
best_params_exist = os.path.exists(best_params_path)

print("====================================================")
print(" XGBOOST HYPERPARAMETER RESOURCE CHECK")
print("====================================================")
print(f"CV scheme: {cv_suffix}")
print(f"Trials file exists: {trials_exist}")
print(f"Best params file exists: {best_params_exist}")
print(f"Running in Colab: {IN_COLAB}")
print("====================================================")


def normalize_xgb_params(params):
    """Convert cached Optuna parameters to XGBoost-compatible Python scalars."""
    params = dict(params)

    # Drop missing values that may appear when reading CSV files.
    params = {
        key: value
        for key, value in params.items()
        if not pd.isna(value)
    }

    int_params = {"n_estimators", "max_depth", "min_child_weight"}
    float_params = {
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "gamma",
        "reg_alpha",
        "reg_lambda"
    }

    for key in int_params.intersection(params):
        params[key] = int(params[key])

    for key in float_params.intersection(params):
        params[key] = float(params[key])

    return params


def load_cached_xgb_hc_optuna_results():
    df_trials = pd.read_csv(optuna_trials_path)
    df_best = pd.read_csv(best_params_path)

    if df_best.empty:
        raise ValueError("Best-params CSV is empty.")

    if "best_mean_weighted_f1" not in df_best.columns:
        raise ValueError("Column 'best_mean_weighted_f1' not found in best-params CSV.")

    best_score = float(df_best.loc[0, "best_mean_weighted_f1"])

    best_params = (
        df_best
        .drop(columns=["best_mean_weighted_f1"])
        .iloc[0]
        .to_dict()
    )
    best_params = normalize_xgb_params(best_params)

    return df_trials, best_params, best_score


def make_xgb_classifier(params, scale_pos_weight=1.0, seed=SEED):
    return xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        booster="gbtree",
        random_state=seed,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        **params
    )


# ------------------------------------------------------------
# Colab: always use cached Optuna results
# ------------------------------------------------------------
if IN_COLAB:
    if not (trials_exist and best_params_exist):
        raise FileNotFoundError(
            "Cached Optuna results were not found in Colab.\n"
            f"Expected trials file: {optuna_trials_path}\n"
            f"Expected best-params file: {best_params_path}\n"
            "Colab execution is configured to use cached Optuna results only."
        )

    df_optuna_trials_xgb_hc, best_xgb_params_hc, best_xgb_score_hc = (
        load_cached_xgb_hc_optuna_results()
    )

    print("====================================================")
    print(" LOADED CACHED XGBOOST HYPERPARAMETERS")
    print("====================================================")
    print(f"Best mean weighted F1-score: {best_xgb_score_hc:.4f}")
    print("Best hyperparameters:")
    for key, value in best_xgb_params_hc.items():
        print(f"{key}: {value}")
    print("====================================================")

# ------------------------------------------------------------
# Local: ask whether to reuse cache or run Optuna
# ------------------------------------------------------------
else:
    reuse_existing_optuna = False
    run_optuna_from_scratch = False

    if trials_exist and best_params_exist:
        user_input = input(
            "Existing Optuna results detected. Reuse them instead of running optimization again? (y/n): "
        ).strip().lower()

        if user_input == "y":
            reuse_existing_optuna = True
        else:
            run_optuna_from_scratch = True
    else:
        print("No complete Optuna result set found. Optimization will be run from scratch.")
        run_optuna_from_scratch = True

    if reuse_existing_optuna:
        df_optuna_trials_xgb_hc, best_xgb_params_hc, best_xgb_score_hc = (
            load_cached_xgb_hc_optuna_results()
        )

        print("====================================================")
        print(" LOADED EXISTING OPTUNA RESULTS")
        print("====================================================")
        print(f"Best mean weighted F1-score: {best_xgb_score_hc:.4f}")
        print("Best hyperparameters:")
        for key, value in best_xgb_params_hc.items():
            print(f"{key}: {value}")
        print("====================================================")

    elif run_optuna_from_scratch:
        import optuna

        optuna.logging.set_verbosity(optuna.logging.WARNING)
        logging.getLogger("xgboost").setLevel(logging.WARNING)

        def objective_xgb_handcrafted(trial):
            candidate_params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "max_depth": trial.suggest_int("max_depth", 2, 10),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "gamma": trial.suggest_float("gamma", 0.0, 5.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            }

            fold_f1 = []

            if USE_GROUPKFOLD:
                cv = GroupKFold(n_splits=n_folds)
                split_iter = cv.split(X_audio_np, y_audio_np, groups_audio_np)
            else:
                cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
                split_iter = cv.split(X_audio_np, y_audio_np)

            for train_idx, test_idx in split_iter:
                X_train = X_audio_np[train_idx]
                X_test = X_audio_np[test_idx]
                y_train = y_audio_np[train_idx]
                y_test = y_audio_np[test_idx]

                n_pos = int(np.sum(y_train == 1))
                n_neg = int(np.sum(y_train == 0))
                scale_pos_weight = (n_neg / n_pos) if n_pos > 0 else 1.0

                model = make_xgb_classifier(
                    params=candidate_params,
                    scale_pos_weight=scale_pos_weight,
                    seed=SEED
                )

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                fold_f1.append(f1_score(y_test, y_pred, average="weighted"))

            return float(np.mean(fold_f1))

        study_xgb_hc = optuna.create_study(
            direction="maximize",
            study_name=f"stressid_xgb_handcrafted_optuna_{cv_suffix}"
        )

        study_xgb_hc.optimize(
            objective_xgb_handcrafted,
            n_trials=N_TRIALS,
            show_progress_bar=True
        )

        best_xgb_params_hc = normalize_xgb_params(study_xgb_hc.best_params)
        best_xgb_score_hc = float(study_xgb_hc.best_value)

        df_optuna_trials_xgb_hc = (
            study_xgb_hc
            .trials_dataframe()
            .sort_values("value", ascending=False)
            .reset_index(drop=True)
        )

        df_optuna_trials_xgb_hc.to_csv(optuna_trials_path, index=False)

        pd.DataFrame([best_xgb_params_hc]).assign(
            best_mean_weighted_f1=best_xgb_score_hc
        ).to_csv(best_params_path, index=False)

        print("====================================================")
        print(" BEST OPTUNA RESULT - XGBOOST HANDCRAFTED FEATURES")
        print("====================================================")
        print(f"Best mean weighted F1-score: {best_xgb_score_hc:.4f}")
        print("Best hyperparameters:")
        for key, value in best_xgb_params_hc.items():
            print(f"{key}: {value}")
        print("====================================================")
        print("\nSaved:")
        print(f"- {optuna_trials_path}")
        print(f"- {best_params_path}")


## 2.6. Tuned XGBoost Evaluation

This cell evaluates the tuned XGBoost configuration obtained in the previous step. The model is tested under the same two validation schemes used for the handcrafted baselines: stratified K-fold and subject-independent GroupKFold. For each fold, `scale_pos_weight` is recomputed from the training partition only, avoiding leakage from the validation fold. The resulting weighted F1-score, accuracy, and balanced accuracy are exported so they can be included in the final comparison tables.

Because the hyperparameters are selected on the same dataset rather than inside a nested cross-validation loop, these results should be interpreted as model-selection results rather than fully independent generalization estimates.


In [ ]:
# ============================================================
# 2.6. Tuned XGBoost Evaluation
# ============================================================

from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, GroupKFold

assert "best_xgb_params_hc" in locals(), "best_xgb_params_hc not found. Run the Optuna tuning cell first."
assert "make_xgb_classifier" in locals(), "make_xgb_classifier not found. Run the Optuna tuning cell first."

best_xgb_params_hc = normalize_xgb_params(best_xgb_params_hc)


def evaluate_xgb_cv(X, y, best_params, cv, groups=None, seed=SEED):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    groups = None if groups is None else np.asarray(groups)

    f1_scores = []
    acc_scores = []
    bacc_scores = []

    if groups is None:
        split_iter = cv.split(X, y)
    else:
        split_iter = cv.split(X, y, groups)

    for train_idx, test_idx in split_iter:
        X_train = X[train_idx]
        X_test = X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]

        n_pos = int(np.sum(y_train == 1))
        n_neg = int(np.sum(y_train == 0))
        scale_pos_weight = (n_neg / n_pos) if n_pos > 0 else 1.0

        model = make_xgb_classifier(
            params=best_params,
            scale_pos_weight=scale_pos_weight,
            seed=seed
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        f1_scores.append(f1_score(y_test, y_pred, average="weighted"))
        acc_scores.append(accuracy_score(y_test, y_pred))
        bacc_scores.append(balanced_accuracy_score(y_test, y_pred))

    return summarize_fold_scores(
        f1_scores=f1_scores,
        acc_scores=acc_scores,
        bacc_scores=bacc_scores
    )


def print_metric_block(title, results):
    print("====================================================")
    print(title)
    print("====================================================")
    print(f"Mean weighted F1:     {results['f1_mean']:.4f}")
    print(f"Std  weighted F1:     {results['f1_std']:.4f}")
    print(f"Mean accuracy:        {results['acc_mean']:.4f}")
    print(f"Std  accuracy:        {results['acc_std']:.4f}")
    print(f"Mean balanced acc.:   {results['bacc_mean']:.4f}")
    print(f"Std  balanced acc.:   {results['bacc_std']:.4f}")


# ------------------------------------------------------------
# Stratified CV
# ------------------------------------------------------------
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

xgb_hc_strat_results = evaluate_xgb_cv(
    X=X_audio,
    y=y_audio,
    best_params=best_xgb_params_hc,
    cv=skf,
    groups=None,
    seed=SEED
)

print_metric_block(
    " XGBOOST HANDCRAFTED FEATURES - STRATIFIED CV",
    xgb_hc_strat_results
)

# ------------------------------------------------------------
# Subject-independent CV
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=n_folds)

xgb_hc_group_results = evaluate_xgb_cv(
    X=X_audio,
    y=y_audio,
    best_params=best_xgb_params_hc,
    cv=gkf,
    groups=groups_audio,
    seed=SEED
)

print()
print_metric_block(
    " XGBOOST HANDCRAFTED FEATURES - SUBJECT-INDEPENDENT CV",
    xgb_hc_group_results
)

# ------------------------------------------------------------
# Export tuned XGBoost results
# ------------------------------------------------------------
eval_rows = [
    {
        "feature_type": "Handcrafted",
        "model": "XGBoost (Optuna-tuned)",
        "cv_scheme": "stratifiedkfold",
        **xgb_hc_strat_results
    },
    {
        "feature_type": "Handcrafted",
        "model": "XGBoost (Optuna-tuned)",
        "cv_scheme": "groupkfold",
        **xgb_hc_group_results
    }
]

df_xgb_eval = pd.DataFrame(eval_rows)

eval_path = os.path.join(
    EVAL_DIR,
    "stressid_xgb_handcrafted_tuned_evaluation_summary.csv"
)

df_xgb_eval.to_csv(eval_path, index=False)

print("\nSaved:")
print(f"- {eval_path}")


## 2.7. Feature Importance Analysis

This cell estimates which handcrafted descriptors contribute most strongly to the tuned XGBoost classifier. The 140 feature positions are mapped back to interpretable descriptor names and broader acoustic families. Feature importances are computed fold by fold using the selected validation scheme and then averaged, which is more stable than fitting a single model once on the full dataset.

The analysis is descriptive: it helps interpret the learned decision patterns, but it is not used as an additional benchmark result.


In [ ]:
# ============================================================
# 2.7. Feature Importance Analysis
# ============================================================

assert "best_xgb_params_hc" in locals(), "best_xgb_params_hc not found. Run the Optuna tuning cell first."
assert "make_xgb_classifier" in locals(), "make_xgb_classifier not found. Run the Optuna tuning cell first."
assert "X_audio" in locals(), "X_audio not found."
assert "y_audio" in locals(), "y_audio not found."
assert X_audio.shape[1] == 140, f"Expected 140 handcrafted features, got {X_audio.shape[1]}."

# ------------------------------------------------------------
# Feature-name mapping
# ------------------------------------------------------------
feature_names_hc = []

feature_names_hc += [f"mfcc_c{i}_mean" for i in range(13)]
feature_names_hc += [f"mfcc_c{i}_std" for i in range(13)]

feature_names_hc += [f"mfcc_delta_c{i}_mean" for i in range(13)]
feature_names_hc += [f"mfcc_delta_c{i}_std" for i in range(13)]

feature_names_hc += [f"mfcc_delta2_c{i}_mean" for i in range(13)]
feature_names_hc += [f"mfcc_delta2_c{i}_std" for i in range(13)]

feature_names_hc += ["spectral_centroid_mean", "spectral_centroid_std"]
feature_names_hc += ["spectral_bandwidth_mean", "spectral_bandwidth_std"]
feature_names_hc += ["spectral_flatness_mean", "spectral_flatness_std"]

feature_names_hc += [f"spectral_contrast_band{i}_mean" for i in range(7)]
feature_names_hc += [f"spectral_contrast_band{i}_std" for i in range(7)]

feature_names_hc += ["spectral_rolloff_mean", "spectral_rolloff_std"]

feature_names_hc += [f"tonnetz_dim{i}_mean" for i in range(6)]
feature_names_hc += [f"tonnetz_dim{i}_std" for i in range(6)]

feature_names_hc += ["zcr_mean", "zcr_std"]

feature_names_hc += [f"tempogram_ratio_bin{i}_mean" for i in range(13)]
feature_names_hc += [f"tempogram_ratio_bin{i}_std" for i in range(13)]

assert len(feature_names_hc) == 140

feature_groups_hc = (
    ["MFCC"] * 13 +
    ["MFCC"] * 13 +
    ["MFCC-delta"] * 13 +
    ["MFCC-delta"] * 13 +
    ["MFCC-delta2"] * 13 +
    ["MFCC-delta2"] * 13 +
    ["Spectral centroid"] * 2 +
    ["Spectral bandwidth"] * 2 +
    ["Spectral flatness"] * 2 +
    ["Spectral contrast"] * 7 +
    ["Spectral contrast"] * 7 +
    ["Spectral rolloff"] * 2 +
    ["Tonnetz"] * 6 +
    ["Tonnetz"] * 6 +
    ["ZCR"] * 2 +
    ["Tempogram ratio"] * 13 +
    ["Tempogram ratio"] * 13
)

assert len(feature_groups_hc) == 140

# ------------------------------------------------------------
# Cross-validation scheme for interpretability
# ------------------------------------------------------------
USE_GROUPKFOLD_IMPORTANCE = True

X_audio_np = np.asarray(X_audio, dtype=np.float32)
y_audio_np = np.asarray(y_audio, dtype=np.int64)

if USE_GROUPKFOLD_IMPORTANCE:
    assert "groups_audio" in locals(), "groups_audio not found, but USE_GROUPKFOLD_IMPORTANCE=True."
    groups_audio_np = np.asarray(groups_audio)
    cv = GroupKFold(n_splits=n_folds)
    split_iter = cv.split(X_audio_np, y_audio_np, groups_audio_np)
    cv_suffix = "groupkfold"
else:
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    split_iter = cv.split(X_audio_np, y_audio_np)
    cv_suffix = "stratifiedkfold"

# ------------------------------------------------------------
# Fold-wise importances
# ------------------------------------------------------------
fold_rows = []

for fold_idx, (train_idx, _) in enumerate(split_iter, start=1):
    X_train = X_audio_np[train_idx]
    y_train = y_audio_np[train_idx]

    n_pos = int(np.sum(y_train == 1))
    n_neg = int(np.sum(y_train == 0))
    scale_pos_weight = (n_neg / n_pos) if n_pos > 0 else 1.0

    model = make_xgb_classifier(
        params=best_xgb_params_hc,
        scale_pos_weight=scale_pos_weight,
        seed=SEED
    )

    model.fit(X_train, y_train)

    df_fold = pd.DataFrame({
        "fold": fold_idx,
        "feature_idx": np.arange(140),
        "feature_name": feature_names_hc,
        "feature_group": feature_groups_hc,
        "importance": model.feature_importances_
    })

    fold_rows.append(df_fold)

df_importance_folds = pd.concat(fold_rows, ignore_index=True)

# ------------------------------------------------------------
# Individual feature summary
# ------------------------------------------------------------
df_importance = (
    df_importance_folds
    .groupby(["feature_idx", "feature_name", "feature_group"], as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std")
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("====================================================")
print(" TOP 20 INDIVIDUAL HANDCRAFTED FEATURES")
print("====================================================")
print(df_importance.head(20).to_string(index=False))

# ------------------------------------------------------------
# Aggregated family importance
# ------------------------------------------------------------
df_group_importance = (
    df_importance
    .groupby("feature_group", as_index=False)["importance_mean"]
    .sum()
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

print("\n====================================================")
print(" AGGREGATED IMPORTANCE BY FEATURE FAMILY")
print("====================================================")
print(df_group_importance.to_string(index=False))

# ------------------------------------------------------------
# Plots
# ------------------------------------------------------------
TOP_N = 20

df_top = df_importance.head(TOP_N).iloc[::-1]

plt.figure(figsize=(10, 8))
plt.barh(df_top["feature_name"], df_top["importance_mean"])
plt.xlabel("Mean XGBoost feature importance across folds")
plt.ylabel("Feature")
plt.title(f"Top {TOP_N} Most Important Handcrafted Features")
plt.tight_layout()
plt.show()

df_group_plot = df_group_importance.iloc[::-1]

plt.figure(figsize=(10, 6))
plt.barh(df_group_plot["feature_group"], df_group_plot["importance_mean"])
plt.xlabel("Summed mean XGBoost feature importance across folds")
plt.ylabel("Feature family")
plt.title("Aggregated Importance by Handcrafted Feature Family")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Exports
# ------------------------------------------------------------
importance_folds_path = os.path.join(
    IMPORTANCE_DIR,
    f"stressid_xgb_handcrafted_feature_importance_folds_{cv_suffix}.csv"
)

importance_individual_path = os.path.join(
    IMPORTANCE_DIR,
    f"stressid_xgb_handcrafted_feature_importance_individual_{cv_suffix}.csv"
)

importance_grouped_path = os.path.join(
    IMPORTANCE_DIR,
    f"stressid_xgb_handcrafted_feature_importance_grouped_{cv_suffix}.csv"
)

df_importance_folds.to_csv(importance_folds_path, index=False)
df_importance.to_csv(importance_individual_path, index=False)
df_group_importance.to_csv(importance_grouped_path, index=False)

print("\nSaved:")
print(f"- {importance_folds_path}")
print(f"- {importance_individual_path}")
print(f"- {importance_grouped_path}")


# 3. Wav2Vec 2.0 pipeline

## 3.1. Wav2Vec 2.0 Embedding Extraction Methodology

This cell defines the Wav2Vec 2.0 embedding utilities used for the deep-representation experiments. Each recording is first processed with the same StressID VAD routine used by the handcrafted pipeline. The resulting speech signal is passed through the Fairseq Wav2Vec model, and temporal mean and standard deviation statistics are computed over the aggregated representation. Concatenating these statistics yields a fixed-length embedding suitable for conventional classifiers.

Model loading is optional. When the Fairseq checkpoint or dependency is unavailable, the notebook can still proceed from cached Wav2Vec embeddings.


In [ ]:
# ============================================================
# 3.1. Wav2Vec 2.0 Embedding Extraction Utilities
# ============================================================

import torch

W2Vmodel = None
CAN_COMPUTE_W2V = False


def DeepAudioFeatures(audio_path, W2Vmodel):
    speech, sr = librosa.load(audio_path, sr=None, mono=True)
    speech_vad = apply_stressid_vad(speech, sr)

    model = W2Vmodel[0] if isinstance(W2Vmodel, (list, tuple)) else W2Vmodel
    model.eval()

    waveform = torch.from_numpy(speech_vad).float().unsqueeze(0)

    with torch.no_grad():
        z = model.feature_extractor(waveform)
        z = model.feature_aggregator(z)

    w2v = z.cpu().numpy()
    w2v_mean = np.mean(w2v, axis=2)
    w2v_std = np.std(w2v, axis=2)

    return w2v_mean, w2v_std, w2v


def DeepAudioEmbedding(audio_path, W2Vmodel):
    w2v_mean, w2v_std, _ = DeepAudioFeatures(audio_path, W2Vmodel)

    embedding = np.concatenate(
        [w2v_mean.squeeze(0), w2v_std.squeeze(0)],
        axis=0
    )

    return embedding.astype(np.float32)


# ------------------------------------------------------------
# Optional Wav2Vec model loading
# ------------------------------------------------------------
checkpoint_path = WAV2VEC_LARGE_PT

if IN_COLAB:
    print("Colab environment detected.")
    print("Skipping Fairseq/Wav2Vec model loading.")
    print("Wav2Vec embeddings will be loaded from cached parquet files.")

elif not os.path.exists(checkpoint_path):
    print("Local Wav2Vec checkpoint not found.")
    print(f"Expected checkpoint path: {checkpoint_path}")
    print("Wav2Vec embeddings will be loaded from cached parquet files.")

else:
    try:
        import fairseq

        print("Loading Wav2Vec model...")
        print(f"Checkpoint: {checkpoint_path}")

        model, cfg, task = fairseq.checkpoint_utils.load_model_ensemble_and_task(
            [checkpoint_path]
        )

        W2Vmodel = model
        CAN_COMPUTE_W2V = True

        print("Wav2Vec model loaded successfully.")

    except ModuleNotFoundError:
        print("Fairseq is not installed in this environment.")
        print("Wav2Vec embeddings will be loaded from cached parquet files.")
        W2Vmodel = None
        CAN_COMPUTE_W2V = False

print("CAN_COMPUTE_W2V:", CAN_COMPUTE_W2V)


## 3.2. Dataset-Level Wav2Vec 2.0 Embedding Construction and Caching

This cell builds the dataset-level Wav2Vec 2.0 embedding matrix. For each retained StressID speech sample, the embedding extractor produces one fixed-length vector, which is stacked into `X_w2v`. The corresponding labels and subject identifiers are stored as `y_w2v` and `groups_w2v`. Because Wav2Vec extraction is computationally expensive and depends on a local Fairseq checkpoint, the embeddings are cached as a parquet file and reused when possible.


In [ ]:
# ============================================================
# 3.2. Dataset-Level Wav2Vec 2.0 Embedding Construction and Caching
# ============================================================

W2V_FEATURE_CACHE = os.path.join(CACHE_DIR, "stressid_w2v_embeddings.parquet")
W2V_META_FILE = os.path.join(CACHE_DIR, "stressid_w2v_embeddings_meta.txt")

cache_exists = os.path.exists(W2V_FEATURE_CACHE)
meta_exists = os.path.exists(W2V_META_FILE)

use_cached_w2v = False
compute_w2v_from_scratch = False

# ------------------------------------------------------------
# Decide whether to use cache or recompute
# ------------------------------------------------------------
if cache_exists:
    print("====================================================")
    print(" EXISTING W2V EMBEDDING CACHE DETECTED")
    print("====================================================")

    if meta_exists and not IN_COLAB:
        with open(W2V_META_FILE, "r", encoding="utf-8") as f:
            print(f.read())

    if IN_COLAB:
        print("Colab environment detected. Using cached W2V embeddings automatically.")
        use_cached_w2v = True
    else:
        if CAN_COMPUTE_W2V:
            user_input = input(
                "Reuse cached W2V embeddings instead of recomputing from scratch? (y/n): "
            ).strip().lower()

            if user_input == "y":
                print("Using existing cached W2V embedding dataset.")
                use_cached_w2v = True
            else:
                print(
                    "Cached W2V embeddings will be ignored. "
                    "A new extraction will overwrite the existing cache."
                )
                compute_w2v_from_scratch = True
        else:
            print("Wav2Vec model is not available. Using cached W2V embeddings.")
            use_cached_w2v = True

else:
    if IN_COLAB:
        raise FileNotFoundError(
            "No cached W2V embedding file was found in Colab.\n"
            f"Expected cache path: {W2V_FEATURE_CACHE}\n"
            "Colab execution requires the cached parquet file because Wav2Vec recomputation is disabled there."
        )

    if CAN_COMPUTE_W2V:
        print("No cached W2V embeddings found. Embeddings will be computed from scratch.")
        compute_w2v_from_scratch = True
    else:
        raise FileNotFoundError(
            "No cached W2V embedding file was found, and Wav2Vec recomputation is not available.\n"
            f"Expected cache path: {W2V_FEATURE_CACHE}\n"
            "Run this notebook locally with Fairseq and a valid Wav2Vec checkpoint, or provide the cached parquet file."
        )

# ------------------------------------------------------------
# Load cached W2V embeddings
# ------------------------------------------------------------
if use_cached_w2v:
    df_w2v, X_w2v, y_w2v, groups_w2v = load_feature_cache(W2V_FEATURE_CACHE)

    print("====================================================")
    print(" W2V EMBEDDING DATASET READY")
    print("====================================================")
    print("Source: cache")
    print(f"Samples: {X_w2v.shape[0]}")
    print(f"Embedding dim: {X_w2v.shape[1]}")
    print(f"Positive class ratio (y=1): {float(np.mean(y_w2v)):.3f}")
    print("====================================================")

# ------------------------------------------------------------
# Compute W2V embeddings from scratch
# ------------------------------------------------------------
elif compute_w2v_from_scratch:
    required_cols = {"filepath", "subject", "binary-stress"}

    assert "df_sid" in locals(), "df_sid not found. Run the corpus-ingestion cell first."
    assert required_cols.issubset(set(df_sid.columns)), (
        f"df_sid must contain columns: {sorted(required_cols)}"
    )
    assert "W2Vmodel" in locals() and W2Vmodel is not None, (
        "W2Vmodel not found. Load the Fairseq Wav2Vec model first."
    )
    assert "DeepAudioEmbedding" in locals(), (
        "DeepAudioEmbedding not found. Run the Wav2Vec utility cell first."
    )

    X_w2v_list = []
    y_w2v_list = []
    groups_w2v_list = []
    failed_files = []

    for _, row in tqdm.tqdm(
        df_sid.iterrows(),
        total=len(df_sid),
        desc="Extracting W2V embeddings"
    ):
        audio_path = row["filepath"]
        label = int(row["binary-stress"])
        subject = row["subject"]

        try:
            embedding = DeepAudioEmbedding(audio_path, W2Vmodel)
            X_w2v_list.append(embedding)
            y_w2v_list.append(label)
            groups_w2v_list.append(subject)
        except Exception as exc:
            failed_files.append((audio_path, str(exc)))

    n_ok = len(X_w2v_list)
    n_fail = len(failed_files)

    print("====================================================")
    print(" W2V EMBEDDING EXTRACTION SUMMARY")
    print("====================================================")
    print(f"Total rows in df_sid: {len(df_sid)}")
    print(f"Successful extractions: {n_ok}")
    print(f"Failed extractions: {n_fail}")

    if n_fail > 0:
        print("\nFirst 10 failures:")
        for failed_path, error_message in failed_files[:10]:
            print(f" - {failed_path}\n   {error_message}")

    assert n_ok > 0, "No W2V embeddings were extracted successfully. See failures above."

    X_w2v = np.vstack(X_w2v_list).astype(np.float32)
    y_w2v = np.array(y_w2v_list, dtype=np.int64)
    groups_w2v = np.array(groups_w2v_list, dtype=object)

    feature_cols = [f"f_{i}" for i in range(X_w2v.shape[1])]
    df_w2v = pd.DataFrame(X_w2v, columns=feature_cols)
    df_w2v["label"] = y_w2v
    df_w2v["subject"] = groups_w2v

    df_w2v.to_parquet(W2V_FEATURE_CACHE, index=False)

    extraction_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    meta_text = (
        f"Last extraction timestamp: {extraction_timestamp}\n"
        f"Samples: {len(df_w2v)}\n"
        f"Embedding dimension: {X_w2v.shape[1]}\n"
        f"Positive ratio: {float(np.mean(y_w2v)):.3f}\n"
        f"Failed files: {len(failed_files)}\n"
        f"Cache file: {W2V_FEATURE_CACHE}\n"
    )

    with open(W2V_META_FILE, "w", encoding="utf-8") as f:
        f.write(meta_text)

    print("\n====================================================")
    print(" W2V EMBEDDING DATASET READY")
    print("====================================================")
    print("Source: fresh extraction")
    print(f"Samples: {X_w2v.shape[0]}")
    print(f"Embedding dim: {X_w2v.shape[1]}")
    print(f"Positive class ratio (y=1): {float(np.mean(y_w2v)):.3f}")
    print(f"Failed files: {len(failed_files)}")
    print("====================================================")
    print(meta_text)


## 3.3. Baseline Evaluation on Wav2Vec 2.0 Embeddings

This cell evaluates the Wav2Vec 2.0 embeddings with the same baseline validation logic used for the handcrafted descriptors. The embeddings from Section 3.2 are used as input to kNN with k = 5 and to a linear SVM classifier, using fold-local feature scaling in both cases. Results are computed under both stratified K-fold and subject-independent GroupKFold so that the deep representation can be compared directly with the handcrafted feature set.

In [ ]:
# ============================================================
# 3.3. Baseline Evaluation on Wav2Vec 2.0 Embeddings
# ============================================================

assert "evaluate_models_cv" in locals(), "evaluate_models_cv not found. Run Section 2.3 first."

if not all(name in locals() for name in ["X_w2v", "y_w2v", "groups_w2v"]):
    assert os.path.exists(W2V_FEATURE_CACHE), f"Missing W2V cache file: {W2V_FEATURE_CACHE}"
    df_w2v, X_w2v, y_w2v, groups_w2v = load_feature_cache(W2V_FEATURE_CACHE)
    w2v_source = "cache"
else:
    w2v_source = "memory"

print("====================================================")
print(" W2V EMBEDDING DATASET READY")
print("====================================================")
print(f"Source: {w2v_source}")
print(f"Samples: {X_w2v.shape[0]}")
print(f"Embedding dim: {X_w2v.shape[1]}")
print(f"Positive class ratio (y=1): {float(np.mean(y_w2v)):.3f}")
print("====================================================")

# ------------------------------------------------------------
# Baseline classifiers
# ------------------------------------------------------------
K_neighbors = KNN_N_NEIGHBORS if "KNN_N_NEIGHBORS" in globals() else 5
SVM_kernel = SVM_KERNEL if "SVM_KERNEL" in globals() else "linear"

w2v_models = {
    f"kNN (k={K_neighbors})": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=K_neighbors))
    ]),
    f"SVM ({SVM_kernel})": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel=SVM_kernel, C=1.0, class_weight="balanced"))
    ])
}

print("Models initialized:")
for name in w2v_models:
    print("-", name)

# ------------------------------------------------------------
# Stratified CV
# ------------------------------------------------------------
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

w2v_strat_results = evaluate_models_cv(
    X=X_w2v,
    y=y_w2v,
    models=w2v_models,
    cv=skf,
    groups=None
)

print("====================================================")
print(" W2V EMBEDDINGS - STRATIFIED 5-FOLD CV")
print("====================================================")
for name, result in w2v_strat_results.items():
    print(name)
    print(f"Mean weighted F1:     {result['f1_mean']:.4f}")
    print(f"Std  weighted F1:     {result['f1_std']:.4f}")
    print(f"Mean accuracy:        {result['acc_mean']:.4f}")
    print(f"Std  accuracy:        {result['acc_std']:.4f}")
    print(f"Mean balanced acc.:   {result['bacc_mean']:.4f}")
    print(f"Std  balanced acc.:   {result['bacc_std']:.4f}")
    print()

# ------------------------------------------------------------
# Subject-independent CV
# ------------------------------------------------------------
gkf = GroupKFold(n_splits=n_folds)

w2v_group_results = evaluate_models_cv(
    X=X_w2v,
    y=y_w2v,
    models=w2v_models,
    cv=gkf,
    groups=groups_w2v
)

print("====================================================")
print(" W2V EMBEDDINGS - SUBJECT-INDEPENDENT CV")
print("====================================================")
for name, result in w2v_group_results.items():
    print(name)
    print(f"Mean weighted F1:     {result['f1_mean']:.4f}")
    print(f"Std  weighted F1:     {result['f1_std']:.4f}")
    print(f"Mean accuracy:        {result['acc_mean']:.4f}")
    print(f"Std  accuracy:        {result['acc_std']:.4f}")
    print(f"Mean balanced acc.:   {result['bacc_mean']:.4f}")
    print(f"Std  balanced acc.:   {result['bacc_std']:.4f}")
    print()


# 4. Comparative Results Summary

## 4.1. Stratified Cross-Validation Results

This cell builds a consolidated stratified-validation table. It combines the StressID paper values, the locally reproduced handcrafted baselines, the tuned handcrafted XGBoost model, and the local Wav2Vec 2.0 baselines when those variables are available in memory. The table reports weighted F1-score, accuracy, and balanced accuracy whenever each metric is available.

The purpose of this table is to summarize sample-level performance under a class-balanced fold construction. It is useful for checking whether the local implementation is broadly consistent with the reported baseline and for comparing feature representations under the same local metric definitions.


In [ ]:
# ============================================================
# 4.1. Stratified Cross-Validation Results
# ============================================================

def fmt(mean, std):
    return f"{mean:.4f} ± {std:.4f}"


knn_label = KNN_MODEL_LABEL if "KNN_MODEL_LABEL" in globals() else "kNN (k=5)"
svm_label = SVM_MODEL_LABEL if "SVM_MODEL_LABEL" in globals() else "SVM (linear)"


stratified_rows = []

# ------------------------------------------------------------
# Authors' reported results (StressID paper, 2-class audio)
# ------------------------------------------------------------
stratified_rows.extend([
    {
        "Result Source": "StressID paper",
        "Feature Type": "Handcrafted",
        "Model": knn_label,
        "Weighted F1-score": "0.6700 ± 0.0600",
        "Accuracy": "0.6000 ± 0.0500",
        "Balanced Accuracy": "—"
    },
    {
        "Result Source": "StressID paper",
        "Feature Type": "Handcrafted",
        "Model": svm_label,
        "Weighted F1-score": "0.6100 ± 0.0600",
        "Accuracy": "0.5400 ± 0.0300",
        "Balanced Accuracy": "—"
    },
    {
        "Result Source": "StressID paper",
        "Feature Type": "Wav2Vec 2.0",
        "Model": "Wav2Vec 2.0 classifier",
        "Weighted F1-score": "0.7000 ± 0.0200",
        "Accuracy": "0.6600 ± 0.0300",
        "Balanced Accuracy": "—"
    }
])

# ------------------------------------------------------------
# Local handcrafted baselines
# ------------------------------------------------------------
if "hc_baseline_strat_results" in locals():
    stratified_rows.extend([
        {
            "Result Source": "Local",
            "Feature Type": "Handcrafted",
            "Model": knn_label,
            "Weighted F1-score": fmt(hc_baseline_strat_results["knn_f1_mean"], hc_baseline_strat_results["knn_f1_std"]),
            "Accuracy": fmt(hc_baseline_strat_results["knn_acc_mean"], hc_baseline_strat_results["knn_acc_std"]),
            "Balanced Accuracy": fmt(hc_baseline_strat_results["knn_bacc_mean"], hc_baseline_strat_results["knn_bacc_std"])
        },
        {
            "Result Source": "Local",
            "Feature Type": "Handcrafted",
            "Model": svm_label,
            "Weighted F1-score": fmt(hc_baseline_strat_results["svm_f1_mean"], hc_baseline_strat_results["svm_f1_std"]),
            "Accuracy": fmt(hc_baseline_strat_results["svm_acc_mean"], hc_baseline_strat_results["svm_acc_std"]),
            "Balanced Accuracy": fmt(hc_baseline_strat_results["svm_bacc_mean"], hc_baseline_strat_results["svm_bacc_std"])
        }
    ])
else:
    print("Warning: hc_baseline_strat_results not found. Local handcrafted baseline rows omitted.")

# ------------------------------------------------------------
# Local tuned XGBoost
# ------------------------------------------------------------
if "xgb_hc_strat_results" in locals():
    stratified_rows.append({
        "Result Source": "Local",
        "Feature Type": "Handcrafted",
        "Model": "XGBoost (Optuna-tuned)",
        "Weighted F1-score": fmt(xgb_hc_strat_results["f1_mean"], xgb_hc_strat_results["f1_std"]),
        "Accuracy": fmt(xgb_hc_strat_results["acc_mean"], xgb_hc_strat_results["acc_std"]),
        "Balanced Accuracy": fmt(xgb_hc_strat_results["bacc_mean"], xgb_hc_strat_results["bacc_std"])
    })
else:
    print("Warning: xgb_hc_strat_results not found. XGBoost stratified row omitted.")

# ------------------------------------------------------------
# Local Wav2Vec 2.0 models
# ------------------------------------------------------------
if "w2v_strat_results" in locals():
    for name, res in w2v_strat_results.items():
        stratified_rows.append({
            "Result Source": "Local",
            "Feature Type": "Wav2Vec 2.0",
            "Model": name,
            "Weighted F1-score": fmt(res["f1_mean"], res["f1_std"]),
            "Accuracy": fmt(res["acc_mean"], res["acc_std"]),
            "Balanced Accuracy": fmt(res["bacc_mean"], res["bacc_std"])
        })
else:
    print("Warning: w2v_strat_results not found. Local Wav2Vec rows omitted.")

df_results_stratified = pd.DataFrame(stratified_rows)

print("====================================================")
print(" 4.1. STRATIFIED CROSS-VALIDATION RESULTS")
print("====================================================")
print(df_results_stratified.to_string(index=False))

display(df_results_stratified)


## 4.2. Subject-Independent Results

This cell builds the corresponding subject-independent comparison table. In this protocol, each subject is assigned to only one fold, so recordings from the same speaker cannot appear in both training and validation partitions. This makes the evaluation more demanding and more informative for practical stress-recognition scenarios involving unseen speakers.

Only local results are included here, because the StressID paper values used above are not subject-independent GroupKFold results in this notebook. Weighted F1-score, accuracy, and balanced accuracy are reported jointly to expose both overall performance and class-sensitive behavior.


In [ ]:
# ============================================================
# 4.2. Subject-Independent Results
# ============================================================

knn_label = KNN_MODEL_LABEL if "KNN_MODEL_LABEL" in globals() else "kNN (k=5)"
svm_label = SVM_MODEL_LABEL if "SVM_MODEL_LABEL" in globals() else "SVM (linear)"

group_rows = []

# ------------------------------------------------------------
# Local handcrafted baselines
# ------------------------------------------------------------
if "hc_baseline_group_results" in locals():
    group_rows.extend([
        {
            "Result Source": "Local",
            "Feature Type": "Handcrafted",
            "Model": knn_label,
            "Weighted F1-score": fmt(hc_baseline_group_results["knn_f1_mean"], hc_baseline_group_results["knn_f1_std"]),
            "Accuracy": fmt(hc_baseline_group_results["knn_acc_mean"], hc_baseline_group_results["knn_acc_std"]),
            "Balanced Accuracy": fmt(hc_baseline_group_results["knn_bacc_mean"], hc_baseline_group_results["knn_bacc_std"])
        },
        {
            "Result Source": "Local",
            "Feature Type": "Handcrafted",
            "Model": svm_label,
            "Weighted F1-score": fmt(hc_baseline_group_results["svm_f1_mean"], hc_baseline_group_results["svm_f1_std"]),
            "Accuracy": fmt(hc_baseline_group_results["svm_acc_mean"], hc_baseline_group_results["svm_acc_std"]),
            "Balanced Accuracy": fmt(hc_baseline_group_results["svm_bacc_mean"], hc_baseline_group_results["svm_bacc_std"])
        }
    ])
else:
    print("Warning: hc_baseline_group_results not found. Local handcrafted baseline rows omitted.")

# ------------------------------------------------------------
# Local tuned XGBoost
# ------------------------------------------------------------
if "xgb_hc_group_results" in locals():
    group_rows.append({
        "Result Source": "Local",
        "Feature Type": "Handcrafted",
        "Model": "XGBoost (Optuna-tuned)",
        "Weighted F1-score": fmt(xgb_hc_group_results["f1_mean"], xgb_hc_group_results["f1_std"]),
        "Accuracy": fmt(xgb_hc_group_results["acc_mean"], xgb_hc_group_results["acc_std"]),
        "Balanced Accuracy": fmt(xgb_hc_group_results["bacc_mean"], xgb_hc_group_results["bacc_std"])
    })
else:
    print("Warning: xgb_hc_group_results not found. XGBoost group row omitted.")

# ------------------------------------------------------------
# Local Wav2Vec 2.0 models
# ------------------------------------------------------------
if "w2v_group_results" in locals():
    for name, res in w2v_group_results.items():
        group_rows.append({
            "Result Source": "Local",
            "Feature Type": "Wav2Vec 2.0",
            "Model": name,
            "Weighted F1-score": fmt(res["f1_mean"], res["f1_std"]),
            "Accuracy": fmt(res["acc_mean"], res["acc_std"]),
            "Balanced Accuracy": fmt(res["bacc_mean"], res["bacc_std"])
        })
else:
    print("Warning: w2v_group_results not found. Local Wav2Vec rows omitted.")

df_results_group = pd.DataFrame(group_rows)

print("====================================================")
print(" 4.2. SUBJECT-INDEPENDENT RESULTS")
print("====================================================")
print(df_results_group.to_string(index=False))

display(df_results_group)


## 4.3. Final Table Export

This cell exports the final comparison tables to CSV files. It adds an explicit evaluation-protocol column to the stratified and subject-independent tables, concatenates them into a single combined dataframe, saves all three files under the evaluation directory, and displays the combined result for quick inspection.


In [ ]:
# ============================================================
# 4.3. Export Final Result Tables
# ============================================================

assert "df_results_stratified" in locals(), "df_results_stratified not found. Run Section 4.1 first."
assert "df_results_group" in locals(), "df_results_group not found. Run Section 4.2 first."

stratified_results_path = os.path.join(
    EVAL_DIR,
    "stressid_final_results_stratified.csv"
)

group_results_path = os.path.join(
    EVAL_DIR,
    "stressid_final_results_subject_independent.csv"
)

combined_results_path = os.path.join(
    EVAL_DIR,
    "stressid_final_results_combined.csv"
)

# Add evaluation protocol column
df_results_stratified_export = df_results_stratified.copy()
df_results_stratified_export.insert(0, "Evaluation Protocol", "Stratified K-Fold")

df_results_group_export = df_results_group.copy()
df_results_group_export.insert(0, "Evaluation Protocol", "Subject-Independent GroupKFold")

# Combined table
df_results_combined = pd.concat(
    [df_results_stratified_export, df_results_group_export],
    ignore_index=True
)

# Save CSV files
df_results_stratified_export.to_csv(stratified_results_path, index=False)
df_results_group_export.to_csv(group_results_path, index=False)
df_results_combined.to_csv(combined_results_path, index=False)

print("====================================================")
print(" FINAL RESULT TABLES EXPORTED")
print("====================================================")
print(f"- {stratified_results_path}")
print(f"- {group_results_path}")
print(f"- {combined_results_path}")
print("====================================================")

display(df_results_combined)
